# Extended Data Fig. 3b — *BMP4* after BMP4 bead placement

Author: Tianlei He. Ported from `BMP4_level_different_timepoint-20260912.ipynb`.

The panel plots *BMP4* counts per million (CPM) at 0, 7 and 24 h, three replicates each, as bars
(mean ± s.d.) with the replicates as points. CPM values are used exactly as computed by the
sequencing provider; nothing here re-normalizes them.

**Changes from the original notebook**
1. Input is the GEO series GSE347564 CPM table instead of the provider's per-order matrix. Column names
   change accordingly: `KN4J6R_1–3_cpm` → `pSMAD_0h_rep1–3`, `KN4J6R_4–6_cpm` → `pSMAD_7h_rep1–3`,
   `KN4J6R_7–9_cpm` → `pSMAD_24h_rep1–3`.
2. Paths are relative to the repository, with a check that the input table exists; outputs go to `bulkseq/output/`.
3. Removed cells that the panel does not use: the pooled-CPM histogram and threshold table, the
   mean-CPM ≥ 2 filtered-table export (the plot reads the unfiltered row), the housekeeping-gene check,
   two Excel exports, and five alternative statistical tests (none is reported for this panel).
4. Added the last cell, which writes the plotted values to `bulkseq/output/ed3b_plotted_values.tsv`.

In [ ]:
# %pip install pandas matplotlib numpy

from pathlib import Path
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "bulkseq" / "README.md").is_file())
CPM_TABLE = Path(os.environ.get(
    "BULK_CPM_TABLE", REPO / "data" / "GSE347564" / "GSE347564_bulk_RNAseq_cpm_all_samples.tsv.gz"))
OUT = REPO / "bulkseq" / "output"
OUT.mkdir(parents=True, exist_ok=True)

annotation_columns = ["gene_id", "gene_name", "gene_biotype"]

# Three groups of three (pSMAD timepoints; matches mapping-stats labels)
GROUPS = {
    "0 hr pSMAD": ["pSMAD_0h_rep1", "pSMAD_0h_rep2", "pSMAD_0h_rep3"],
    "7 hr pSMAD": ["pSMAD_7h_rep1", "pSMAD_7h_rep2", "pSMAD_7h_rep3"],
    "24 hr pSMAD": ["pSMAD_24h_rep1", "pSMAD_24h_rep2", "pSMAD_24h_rep3"],
}

cpm_cols = [c for cols in GROUPS.values() for c in cols]

TARGET_GENE = "BMP4"
EXPECTED_GENE_ID = "ENSG00000125378"
out_figure_cpm = OUT / "BMP4_samples1-9_three_groups_cpm.png"
out_figure_log2 = OUT / "BMP4_samples1-9_three_groups_log2cpm1.png"

In [ ]:
if not CPM_TABLE.is_file():
    raise FileNotFoundError(f"CPM table not found: {CPM_TABLE}. See bulkseq/README.md.")
df = pd.read_csv(CPM_TABLE, sep="\t")

missing = [c for c in annotation_columns + cpm_cols if c not in df.columns]
if missing:
    raise ValueError(f"Matrix missing columns: {missing}")

plot_df = df[annotation_columns + cpm_cols].copy()
len(plot_df)

In [ ]:
sample_map = [{"group": g, "cpm_column": c} for g, cols in GROUPS.items() for c in cols]
display(pd.DataFrame(sample_map))

hits = plot_df[plot_df["gene_name"].fillna("").str.upper() == TARGET_GENE.upper()]
if hits.empty:
    raise ValueError(f"No row with gene_name == {TARGET_GENE!r}.")
if len(hits) > 1:
    print(f"WARNING: {len(hits)} rows match {TARGET_GENE}; using the first.")
    display(hits[["gene_id", "gene_name", "gene_biotype"] + cpm_cols].head(10))

gene = hits.iloc[0]
print(
    f"Using: gene_id={gene['gene_id']!r}, gene_name={gene['gene_name']!r}, biotype={gene.get('gene_biotype', 'n/a')!r}"
)

if EXPECTED_GENE_ID and str(gene["gene_id"]).upper() != EXPECTED_GENE_ID.upper():
    print(
        f"WARNING: gene_id {gene['gene_id']!r} != EXPECTED_GENE_ID {EXPECTED_GENE_ID!r} — verify annotation."
    )

vals_19 = np.array([float(gene[c]) for c in cpm_cols], dtype=float)
if not np.isfinite(vals_19).all():
    bad = [cpm_cols[i] for i, v in enumerate(vals_19) if not np.isfinite(v)]
    raise ValueError(f"Non-finite CPM in: {bad}")
if (vals_19 < 0).any():
    raise ValueError("Negative CPM values found for target gene.")

In [ ]:
labels = list(GROUPS.keys())
means_cpm = []
stds_cpm = []
means_log2 = []
stds_log2 = []
all_vals_cpm = []
all_vals_log2 = []

for cols in GROUPS.values():
    vals = [float(gene[c]) for c in cols]
    all_vals_cpm.append(vals)
    lv = [np.log2(v + 1.0) for v in vals]
    all_vals_log2.append(lv)
    means_cpm.append(np.mean(vals))
    stds_cpm.append(np.std(vals, ddof=1) if len(vals) > 1 else 0.0)
    means_log2.append(np.mean(lv))
    stds_log2.append(np.std(lv, ddof=1) if len(lv) > 1 else 0.0)

x = np.arange(len(labels))
width = 0.55
colors = ["#1b9e77", "#d95f02", "#7570b3"]


def _bar_replicate_panel(means, stds, all_vals, ylabel, title, path):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(
        x,
        means,
        width,
        yerr=stds,
        capsize=4,
        color=colors,
        edgecolor="black",
        linewidth=0.8,
        alpha=0.9,
    )
    for i, vals in enumerate(all_vals):
        jitter = np.linspace(-0.12, 0.12, len(vals))
        ax.scatter(
            x[i] + jitter,
            vals,
            color="black",
            s=36,
            zorder=3,
            label="Replicates" if i == 0 else None,
        )
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=15, ha="right")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    ax.legend(loc="best")
    fig.tight_layout()
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.savefig(OUT / "BMP4_samples_1-9_three_groups.pdf", format="pdf", dpi=300)
    plt.show()
    print(f"Saved: {path}")


_bar_replicate_panel(
    means_cpm,
    stds_cpm,
    all_vals_cpm,
    f"{TARGET_GENE} (CPM)",
    f"{TARGET_GENE} — KN4J6R samples 1–9 (three timepoint groups)",
    out_figure_cpm,
)

# _bar_replicate_panel(
#     means_log2,
#     stds_log2,
#     all_vals_log2,
#     f"{TARGET_GENE} log2(CPM + 1)",
#     f"{TARGET_GENE} — log2(CPM+1) (same transform as marker analysis)",
#     out_figure_log2,
# )

summary = pd.DataFrame(
    {
        "group": labels,
        "mean_cpm": means_cpm,
        "std_cpm": stds_cpm,
        "mean_log2_cpm1": means_log2,
        "std_log2_cpm1": stds_log2,
        "replicate_cpms": [str(v) for v in all_vals_cpm],
    }
)
display(summary)

In [ ]:
plotted_rows = [(gene["gene_id"], TARGET_GENE, g, c, float(gene[c])) for g, cols in GROUPS.items() for c in cols]

plotted = pd.DataFrame(
    [{"gene_id": row_gene_id, "gene_name": sym, "group": g, "sample": c, "cpm": v}
     for (row_gene_id, sym, g, c, v) in plotted_rows]
)
plotted.to_csv(OUT / "ed3b_plotted_values.tsv", sep="\t", index=False)
print(f"ED 3b: wrote {len(plotted)} plotted values to bulkseq/output/ed3b_plotted_values.tsv")